# Week 1-05 — Scalping / Short-Term Mean-Reversion Strategy (Educational Backtest)

**Important context:** this notebook does **not** reproduce a real, documented algorithm from any video.
The source clip (a repurposed news segment about high-frequency trading, used in a promotional TikTok)
never actually shows entry/exit rules, thresholds, or logic — it's marketing narration, not a strategy spec.

What this notebook *does* implement is a legitimate, standard **scalping-style, short-term mean-reversion
strategy** — the closest real, testable concept to what the clip was gesturing at (fast in-and-out trades
capturing small price moves). Treat this as an educational backtest, not a validated trading system:

- No real market microstructure (queue position, colocation, latency) is modeled — genuine HFT/market-making
  edges depend on all of that.
- Data below is **synthetic** (simulated), so you can run this fully offline. Swap in your own OHLC data
  (from your existing notebooks / data pipeline) by replacing the data-loading cell.
- Commission and slippage are modeled simply — tune them to match your own broker/venue.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)
plt.rcParams["figure.figsize"] = (12, 5)


## 1. Data

Replace this cell with your own data-loading function (e.g. the same loader used in your other
week-N notebooks) if you want to backtest on real historical prices. For now we simulate a
realistic intraday price series with a mean-reverting component layered on a random walk, so the
mean-reversion signal below actually has something to find.


In [ ]:
def generate_synthetic_prices(n_bars=5000, start_price=100.0, bar_seconds=10,
                               drift=0.0, vol=0.0006, mean_reversion_strength=0.02, seed=42):
    """
    Simulate a short-timeframe price series as a random walk with a small
    mean-reverting (Ornstein-Uhlenbeck style) component, so short-term
    dips/spikes tend to partially revert -- similar to what a scalping
    strategy tries to exploit.
    """
    rng = np.random.default_rng(seed)
    prices = np.zeros(n_bars)
    prices[0] = start_price
    fair_value = start_price

    for i in range(1, n_bars):
        fair_value *= (1 + drift + rng.normal(0, vol))
        # pull current price back toward the slowly-evolving fair value
        reversion = mean_reversion_strength * (fair_value - prices[i - 1])
        noise = rng.normal(0, vol) * prices[i - 1]
        prices[i] = prices[i - 1] + reversion + noise

    idx = pd.date_range("2024-01-02 09:30:00", periods=n_bars, freq=f"{bar_seconds}s")
    df = pd.DataFrame({"close": prices}, index=idx)
    df["open"] = df["close"].shift(1).fillna(df["close"].iloc[0])
    df["high"] = df[["open", "close"]].max(axis=1) * (1 + rng.uniform(0, 0.0003, n_bars))
    df["low"] = df[["open", "close"]].min(axis=1) * (1 - rng.uniform(0, 0.0003, n_bars))
    return df

prices = generate_synthetic_prices()
prices.head()


In [ ]:
prices["close"].plot(title="Synthetic Price Series (10s bars)")
plt.ylabel("Price")
plt.show()


## 2. Signal — short-term z-score mean reversion

- Compute a fast rolling mean and rolling standard deviation of price.
- z-score = (price - rolling_mean) / rolling_std
- **Long** when price has dropped meaningfully below its short-term average (z-score below
  `-entry_z`) — betting on a quick bounce back toward the mean.
- **Short** when price has spiked meaningfully above its short-term average (z-score above
  `entry_z`).
- This is the standard formulation of a scalping / mean-reversion signal — tune `lookback`
  and `entry_z` to your instrument's actual noise profile.


In [ ]:
def add_signal(df, lookback=30, entry_z=1.5):
    df = df.copy()
    df["rolling_mean"] = df["close"].rolling(lookback).mean()
    df["rolling_std"] = df["close"].rolling(lookback).std()
    df["zscore"] = (df["close"] - df["rolling_mean"]) / df["rolling_std"]

    df["signal"] = 0
    df.loc[df["zscore"] < -entry_z, "signal"] = 1   # long entry
    df.loc[df["zscore"] > entry_z, "signal"] = -1    # short entry
    return df

prices = add_signal(prices)
prices[["close", "rolling_mean", "zscore", "signal"]].dropna().head()


## 3. Backtest engine

Rules:
- One position open at a time (flat, long, or short) — realistic for a small scalping account.
- **Entry**: on a signal (long or short), enter at the next bar's open.
- **Exit**: whichever comes first —
  - profit target hit (`take_profit_pct`)
  - stop loss hit (`stop_loss_pct`)
  - max holding period reached (`max_hold_bars`) — scalps are meant to be short-lived
- **Position sizing**: fixed fraction of current equity per trade (`risk_fraction`).
- Simple commission + slippage per round-trip trade (`cost_per_trade_pct`).


In [ ]:
def backtest_scalping(df, take_profit_pct=0.0015, stop_loss_pct=0.0010,
                       max_hold_bars=20, risk_fraction=1.0, cost_per_trade_pct=0.0005,
                       starting_equity=10_000.0):
    df = df.dropna(subset=["signal", "zscore"]).reset_index()
    equity = starting_equity
    equity_curve = []
    trades = []

    position = 0        # -1, 0, 1
    entry_price = None
    entry_bar = None
    entry_equity = None

    for i in range(len(df) - 1):
        row = df.iloc[i]
        next_row = df.iloc[i + 1]

        if position == 0 and row["signal"] != 0:
            position = row["signal"]
            entry_price = next_row["open"]
            entry_bar = i + 1
            entry_equity = equity

        elif position != 0:
            bars_held = i - entry_bar
            move_pct = (row["close"] - entry_price) / entry_price * position

            hit_tp = move_pct >= take_profit_pct
            hit_sl = move_pct <= -stop_loss_pct
            timed_out = bars_held >= max_hold_bars

            if hit_tp or hit_sl or timed_out:
                exit_price = next_row["open"]
                realized_pct = (exit_price - entry_price) / entry_price * position
                realized_pct -= cost_per_trade_pct  # round-trip cost

                pnl = entry_equity * risk_fraction * realized_pct
                equity += pnl

                trades.append({
                    "entry_time": df.iloc[entry_bar]["index"],
                    "exit_time": next_row["index"],
                    "direction": "long" if position == 1 else "short",
                    "entry_price": entry_price,
                    "exit_price": exit_price,
                    "return_pct": realized_pct,
                    "pnl": pnl,
                    "exit_reason": "take_profit" if hit_tp else ("stop_loss" if hit_sl else "timeout"),
                })

                position = 0
                entry_price = None
                entry_bar = None
                entry_equity = None

        equity_curve.append(equity)

    equity_curve = pd.Series(equity_curve, index=df["index"].iloc[:len(equity_curve)])
    trades_df = pd.DataFrame(trades)
    return equity_curve, trades_df

equity_curve, trades_df = backtest_scalping(prices)
print(f"Total trades: {len(trades_df)}")
trades_df.head()


## 4. Performance metrics


In [ ]:
def performance_summary(equity_curve, trades_df, starting_equity=10_000.0, bars_per_year=None):
    total_return = equity_curve.iloc[-1] / starting_equity - 1
    running_max = equity_curve.cummax()
    drawdown = equity_curve / running_max - 1
    max_drawdown = drawdown.min()

    if len(trades_df) > 0:
        win_rate = (trades_df["pnl"] > 0).mean()
        avg_win = trades_df.loc[trades_df["pnl"] > 0, "return_pct"].mean()
        avg_loss = trades_df.loc[trades_df["pnl"] <= 0, "return_pct"].mean()
    else:
        win_rate = avg_win = avg_loss = np.nan

    bar_returns = equity_curve.pct_change().dropna()
    if bar_returns.std() > 0:
        # annualize assuming 10s bars, ~6.5h trading day, 252 trading days
        bars_per_day = 6.5 * 60 * 60 / 10
        ann_factor = np.sqrt(bars_per_day * 252)
        sharpe = bar_returns.mean() / bar_returns.std() * ann_factor
    else:
        sharpe = np.nan

    return {
        "Total Return": f"{total_return:.2%}",
        "Max Drawdown": f"{max_drawdown:.2%}",
        "Num Trades": len(trades_df),
        "Win Rate": f"{win_rate:.2%}" if not np.isnan(win_rate) else "n/a",
        "Avg Win": f"{avg_win:.3%}" if not np.isnan(avg_win) else "n/a",
        "Avg Loss": f"{avg_loss:.3%}" if not np.isnan(avg_loss) else "n/a",
        "Sharpe (annualized, approx.)": f"{sharpe:.2f}" if not np.isnan(sharpe) else "n/a",
    }

summary = performance_summary(equity_curve, trades_df)
for k, v in summary.items():
    print(f"{k:35s}: {v}")


## 5. Plots


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

equity_curve.plot(ax=axes[0], title="Equity Curve")
axes[0].set_ylabel("Equity ($)")

running_max = equity_curve.cummax()
drawdown = equity_curve / running_max - 1
drawdown.plot(ax=axes[1], color="crimson", title="Drawdown")
axes[1].set_ylabel("Drawdown")
axes[1].fill_between(drawdown.index, drawdown.values, 0, color="crimson", alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
prices["close"].plot(ax=ax, alpha=0.6, label="Price")

if len(trades_df) > 0:
    longs = trades_df[trades_df["direction"] == "long"]
    shorts = trades_df[trades_df["direction"] == "short"]
    ax.plot(longs["entry_time"], longs["entry_price"], "^", color="green", markersize=6, label="Long entry", linestyle="none")
    ax.plot(shorts["entry_time"], shorts["entry_price"], "v", color="red", markersize=6, label="Short entry", linestyle="none")
    ax.plot(trades_df["exit_time"], trades_df["exit_price"], "x", color="black", markersize=5, label="Exit", linestyle="none")

ax.set_title("Trade Entries / Exits on Price")
ax.legend()
plt.show()


## 6. Parameters to tune

| Parameter | Meaning | Where |
|---|---|---|
| `lookback` | window for rolling mean/std used in z-score | `add_signal` |
| `entry_z` | how extreme a deviation triggers a trade | `add_signal` |
| `take_profit_pct` / `stop_loss_pct` | exit thresholds | `backtest_scalping` |
| `max_hold_bars` | forces a scalp to stay short-lived | `backtest_scalping` |
| `risk_fraction` | fraction of equity risked per trade | `backtest_scalping` |
| `cost_per_trade_pct` | commission + slippage, round trip | `backtest_scalping` |

### Honest caveats
- This is a **mean-reversion scalp**, not the actual (undisclosed) approach referenced in the video —
  nothing in that clip specified real rules.
- Results on synthetic data won't transfer directly to real markets; swap in your own historical data
  and re-tune before drawing any conclusions.
- No modeling of order book depth, queue priority, or latency — real HFT edges live almost entirely in
  those details, which a bar-based backtest like this cannot capture.
